# Sentence Transformer Embeddings and Similarity Analysis

This notebook demonstrates how to use sentence transformers to:
- Convert text into embeddings
- Compare similarity between statements using multiple metrics
- Visualize embedding dimensions and similarity matrices

## 1. Install Dependencies

In [ ]:
# Install required libraries
!pip install sentence-transformers scikit-learn scipy numpy -q

## 2. Import Libraries

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

## 3. Define Helper Functions

In [ ]:
def load_model(model_name="all-MiniLM-L6-v2"):
    """Load a pre-trained sentence transformer model."""
    model = SentenceTransformer(model_name)
    return model

def get_embeddings(model, texts):
    """Convert texts to embeddings."""
    embeddings = model.encode(texts, convert_to_tensor=False)
    return embeddings

def compute_cosine_similarity(embedding1, embedding2):
    """Compute cosine similarity between two embeddings."""
    similarity = cosine_similarity([embedding1], [embedding2])[0][0]
    return similarity

def compute_euclidean_distance(embedding1, embedding2):
    """Compute Euclidean distance between two embeddings."""
    distance = euclidean(embedding1, embedding2)
    return distance

def compute_dot_product(embedding1, embedding2):
    """Compute dot product between two embeddings."""
    dot_product = np.dot(embedding1, embedding2)
    return dot_product

def print_first_n_dimensions(embedding, n=5):
    """Return the first n dimensions of an embedding."""
    return embedding[:n]

print("Helper functions defined!")

## 4. Define Sample Statements

In [ ]:
# Sample statements for comparison
statements = [
    "The cat is sitting on the mat.",
    "A feline is resting on the rug.",
    "The dog is playing in the park.",
    "The weather is beautiful today.",
    "Today is a sunny day with clear skies."
]

print(f"Total statements: {len(statements)}")
print("\nStatements to analyze:")
for i, stmt in enumerate(statements, 1):
    print(f"{i}. {stmt}")

## 5. Load Model and Generate Embeddings

In [ ]:
# Load the model
print("Loading sentence transformer model...")
model = load_model()
print("Model loaded successfully!")

# Generate embeddings for all statements
print("\nGenerating embeddings for statements...")
embeddings = get_embeddings(model, statements)
print(f"Embeddings generated! Shape: {embeddings.shape}")

## 6. Display First Five Dimensions of Embeddings

In [ ]:
print("="*70)
print("FIRST FIVE DIMENSIONS OF EMBEDDINGS")
print("="*70)

for i, (statement, embedding) in enumerate(zip(statements, embeddings)):
    first_five = print_first_n_dimensions(embedding, n=5)
    print(f"\nStatement {i+1}: {statement}")
    print(f"First 5 dimensions: {first_five}")

## 7. Compare Similarity Between Statements

In [ ]:
print("="*70)
print("SIMILARITY SCORES BETWEEN STATEMENTS")
print("="*70)

print(f"\nComparing '{statements[0]}' with other statements:\n")

for j in range(1, len(statements)):
    cosine_sim = compute_cosine_similarity(embeddings[0], embeddings[j])
    euclidean_dist = compute_euclidean_distance(embeddings[0], embeddings[j])
    dot_prod = compute_dot_product(embeddings[0], embeddings[j])

    print(f"Statement {j+1}: {statements[j]}")
    print(f"  Cosine Similarity:    {cosine_sim:.4f}")
    print(f"  Euclidean Distance:   {euclidean_dist:.4f}")
    print(f"  Dot Product:          {dot_prod:.4f}")
    print()

## 8. Pairwise Cosine Similarity Matrix

In [ ]:
# Calculate similarity matrix
similarity_matrix = cosine_similarity(embeddings)

# Display as formatted table
print("="*70)
print("PAIRWISE COSINE SIMILARITY MATRIX")
print("="*70)
print("\n     ", end="")
for i in range(len(statements)):
    print(f"  S{i+1}   ", end="")
print()

for i, row in enumerate(similarity_matrix):
    print(f"S{i+1}: ", end="")
    for val in row:
        print(f"{val:.3f} ", end="")
    print()

## 9. Visualize Similarity Matrix as Heatmap

In [ ]:
# Create a heatmap of the similarity matrix
plt.figure(figsize=(10, 8))

# Create labels for statements
labels = [f"S{i+1}" for i in range(len(statements))]

# Plot heatmap
sns.heatmap(similarity_matrix, 
            annot=True, 
            fmt='.3f', 
            cmap='coolwarm',
            xticklabels=labels,
            yticklabels=labels,
            cbar_kws={'label': 'Cosine Similarity'},
            vmin=0, vmax=1)

plt.title('Pairwise Cosine Similarity Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Statements')
plt.ylabel('Statements')
plt.tight_layout()
plt.show()

## 10. Embedding Statistics

In [ ]:
print("="*70)
print("EMBEDDING STATISTICS")
print("="*70)
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Number of statements: {embeddings.shape[0]}")
print(f"Mean embedding value: {np.mean(embeddings):.4f}")
print(f"Std embedding value:  {np.std(embeddings):.4f}")
print(f"Min embedding value:  {np.min(embeddings):.4f}")
print(f"Max embedding value:  {np.max(embeddings):.4f}")

## 11. Create Similarity DataFrame

In [ ]:
# Create a DataFrame for better visualization
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"S{i+1}" for i in range(len(statements))],
    columns=[f"S{i+1}" for i in range(len(statements))]
)

print("Similarity DataFrame:")
print(similarity_df.round(4))

## 12. Interactive Comparison - Custom Statements

In [ ]:
# Function to compare custom statements
def compare_custom_statements(text1, text2):
    """Compare two custom statements."""
    custom_embeddings = get_embeddings(model, [text1, text2])
    
    cosine_sim = compute_cosine_similarity(custom_embeddings[0], custom_embeddings[1])
    euclidean_dist = compute_euclidean_distance(custom_embeddings[0], custom_embeddings[1])
    dot_prod = compute_dot_product(custom_embeddings[0], custom_embeddings[1])
    
    print(f"\nText 1: {text1}")
    print(f"Text 2: {text2}")
    print(f"\nResults:")
    print(f"  Cosine Similarity:    {cosine_sim:.4f}")
    print(f"  Euclidean Distance:   {euclidean_dist:.4f}")
    print(f"  Dot Product:          {dot_prod:.4f}")
    print(f"  First 5 dims (Text1): {custom_embeddings[0][:5]}")
    print(f"  First 5 dims (Text2): {custom_embeddings[1][:5]}")

# Example usage
compare_custom_statements(
    "I love eating pizza",
    "Pizza is my favorite food"
)

## 13. Summary and Key Insights

In [ ]:
print("="*70)
print("SUMMARY AND KEY INSIGHTS")
print("="*70)

# Find most similar and most dissimilar pairs
np.fill_diagonal(similarity_matrix, -1)  # Exclude diagonal
max_sim_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
min_sim_idx = np.unravel_index(np.argmin(similarity_matrix), similarity_matrix.shape)

max_sim_val = similarity_matrix[max_sim_idx]
min_sim_val = similarity_matrix[min_sim_idx]

print(f"\nMost Similar Pair:")
print(f"  S{max_sim_idx[0]+1}: {statements[max_sim_idx[0]]}")
print(f"  S{max_sim_idx[1]+1}: {statements[max_sim_idx[1]]}")
print(f"  Similarity Score: {max_sim_val:.4f}")

print(f"\nMost Dissimilar Pair:")
print(f"  S{min_sim_idx[0]+1}: {statements[min_sim_idx[0]]}")
print(f"  S{min_sim_idx[1]+1}: {statements[min_sim_idx[1]]}")
print(f"  Similarity Score: {min_sim_val:.4f}")

print(f"\nModel Information:")
print(f"  Model Name: all-MiniLM-L6-v2")
print(f"  Embedding Dimension: {embeddings.shape[1]}")
print(f"  Pooling Strategy: Mean pooling")